# Pedigree Selection Tutorial - Clonal Breeding

This notebook replicates the AlphaSimR clonal breeding pedigree selection tutorial using AlphaSimPy.
It demonstrates pedigree-based selection in a clonal tea breeding program where pedigree BLUP is used
to predict breeding values and skip early evaluation stages (HPT1-3).

**Authors**: Translated from AlphaSimR tutorial by Nelson Lubanga, Gregor Gorjanc, Jon Bancic, Philip Greenspoon, Chris Gaynor  
**Date**: 2024  
**Package**: AlphaSimPy

## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import solve
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import spsolve
from AlphaSimPy import (
    runMacs2, SimParam, newPop, randCross, setPheno, selectInd,
    meanG, varG, mergePops
)

print("AlphaSimPy Clonal Breeding - Pedigree Selection Tutorial")
print("All libraries imported successfully!")

## Global Parameters

Set up the simulation parameters for the clonal breeding program.

In [ ]:
# Number of simulation replications and breeding cycles
n_reps = 1  # Number of simulation replicates
n_burnin = 40  # Number of years in burnin phase
n_future = 40  # Number of years in future phase
start_records = 35  # Year when training and pedigree record collecting begins
n_cycles = n_burnin + n_future

# Genome simulation
n_chr = 15  # Number of chromosomes
n_qtl = 160  # Number of QTL per chromosome: 15 chr x 160 QTL = 2400 QTLs
n_snp = 600  # Simulate SNP chip with 9000 markers
gen_len = 1  # Genetic length
phy_len = 1e8  # Physical length
mut_rate = 2.5e-8  # Mutation rate

# Initial parents mean and variance
init_mean_g = 2500  # Phenotypic mean
init_var_g = 150000  # Genetic variance
init_var_ge = 150000  # Genotype-by-year interaction variance
var_e = 2800000  # Single variance

# Breeding program details
n_parents = 20  # Number of parents (and founders)
n_crosses = 100  # Number of crosses
n_progeny = 20  # Number of progenies per cross
n_clones_act = 500  # Number of individuals selected at ACT stage
n_clones_ect = 40  # Number of individuals selected at ECT stage

# Effective replication of yield trials
rep_hpt = 1  # h2 = 0.05
rep_act = 15  # h2 = 0.45
rep_ect = 50  # h2 = 0.65

scenario_name = "ClonalPedigree"

print(f"Simulation Parameters:")
print(f"  Replicates: {n_reps}")
print(f"  Burn-in years: {n_burnin}")
print(f"  Future years: {n_future}")
print(f"  Total cycles: {n_cycles}")
print(f"  Chromosomes: {n_chr}")
print(f"  QTL per chromosome: {n_qtl}")
print(f"  SNP per chromosome: {n_snp}")
print(f"  Parents: {n_parents}")
print(f"  Crosses per year: {n_crosses}")
print(f"  Progeny per cross: {n_progeny}")

## Helper Functions for Pedigree BLUP

These functions implement pedigree-based BLUP prediction:
- `build_relationship_matrix`: Constructs the numerator relationship matrix (A) from pedigree
- `solve_pedigree_blup`: Solves the mixed model equations for pedigree BLUP

In [ ]:
def build_relationship_matrix(ped_df):
    """
    Build the numerator relationship matrix (A) from a pedigree dataframe.
    
    Parameters:
    -----------
    ped_df : pandas.DataFrame
        Dataframe with columns ['id', 'sire', 'dam']
        Missing parents should be 0 or NaN
    
    Returns:
    --------
    A : numpy.ndarray
        Numerator relationship matrix
    id_map : dict
        Mapping from original IDs to matrix indices
    """
    # Create a copy and handle missing values
    ped = ped_df.copy()
    ped['sire'] = ped['sire'].fillna(0).astype(int)
    ped['dam'] = ped['dam'].fillna(0).astype(int)
    
    # Create mapping from IDs to indices
    all_ids = sorted(set(ped['id'].tolist() + ped['sire'].tolist() + ped['dam'].tolist()))
    all_ids = [x for x in all_ids if x != 0]  # Remove 0 (unknown parent)
    id_map = {id_val: idx for idx, id_val in enumerate(all_ids)}
    
    n = len(all_ids)
    A = np.eye(n)
    
    # Build A matrix using Henderson's rules
    for idx, row in ped.iterrows():
        if row['id'] not in id_map:
            continue
        
        i = id_map[row['id']]
        sire_idx = id_map.get(row['sire'], -1) if row['sire'] != 0 else -1
        dam_idx = id_map.get(row['dam'], -1) if row['dam'] != 0 else -1
        
        if sire_idx >= 0 and dam_idx >= 0:
            # Both parents known
            A[i, i] = 1.0 + 0.5 * A[sire_idx, dam_idx]
            for j in range(n):
                if j != i:
                    A[i, j] = 0.5 * (A[sire_idx, j] + A[dam_idx, j])
                    A[j, i] = A[i, j]
        elif sire_idx >= 0:
            # Only sire known
            A[i, i] = 1.0
            for j in range(n):
                if j != i:
                    A[i, j] = 0.5 * A[sire_idx, j]
                    A[j, i] = A[i, j]
        elif dam_idx >= 0:
            # Only dam known
            A[i, i] = 1.0
            for j in range(n):
                if j != i:
                    A[i, j] = 0.5 * A[dam_idx, j]
                    A[j, i] = A[i, j]
        else:
            # No parents known (founder)
            A[i, i] = 1.0
    
    return A, id_map


def solve_pedigree_blup(pheno_data, ped_df, var_e=None, var_a=None):
    """
    Solve pedigree BLUP mixed model equations.
    
    Model: y = X*beta + Z*u + e
    where u ~ N(0, A*var_a) and e ~ N(0, I*var_e)
    
    Parameters:
    -----------
    pheno_data : pandas.DataFrame
        Dataframe with columns ['id', 'pheno', 'year'] (year optional)
    ped_df : pandas.DataFrame
        Pedigree dataframe with columns ['id', 'sire', 'dam']
    var_e : float, optional
        Error variance. If None, estimated from data.
    var_a : float, optional
        Additive genetic variance. If None, estimated from data.
    
    Returns:
    --------
    ebv : numpy.ndarray
        Estimated breeding values
    id_order : list
        Order of IDs corresponding to EBV
    """
    # Merge phenotype and pedigree data
    all_ids = sorted(set(pheno_data['id'].tolist() + ped_df['id'].tolist()))
    
    # Build relationship matrix
    A, id_map = build_relationship_matrix(ped_df)
    
    # Create design matrices
    n = len(all_ids)
    n_obs = len(pheno_data)
    
    # Fixed effects matrix X (intercept + year if available)
    if 'year' in pheno_data.columns:
        years = pheno_data['year'].unique()
        n_fixed = len(years) + 1  # intercept + year effects
        X = np.zeros((n_obs, n_fixed))
        X[:, 0] = 1.0  # intercept
        for i, year in enumerate(years):
            X[pheno_data['year'] == year, i + 1] = 1.0
    else:
        n_fixed = 1
        X = np.ones((n_obs, 1))
    
    # Random effects matrix Z
    Z = np.zeros((n_obs, n))
    for idx, row in pheno_data.iterrows():
        if row['id'] in id_map:
            Z[idx, id_map[row['id']]] = 1.0
    
    # Response vector y
    y = pheno_data['pheno'].values.reshape(-1, 1)
    
    # Estimate variance components if not provided
    if var_e is None or var_a is None:
        # Simple method of moments estimates
        if var_e is None:
            var_e = np.var(y) * 0.7  # Rough estimate
        if var_a is None:
            var_a = np.var(y) * 0.3  # Rough estimate
    
    # Mixed model equations
    # [X'X  X'Z] [beta]   [X'y]
    # [Z'X  Z'Z + A^-1*lambda] [u] = [Z'y]
    # where lambda = var_e / var_a
    
    lambda_val = var_e / var_a if var_a > 0 else 1.0
    
    # Build MME coefficient matrix
    XtX = X.T @ X
    XtZ = X.T @ Z
    ZtX = Z.T @ X
    ZtZ = Z.T @ Z
    
    # A inverse (simplified - using pseudo-inverse for numerical stability)
    try:
        A_inv = np.linalg.inv(A)
    except np.linalg.LinAlgError:
        A_inv = np.linalg.pinv(A)
    
    # Build coefficient matrix
    top_left = XtX
    top_right = XtZ
    bottom_left = ZtX
    bottom_right = ZtZ + A_inv * lambda_val
    
    MME_coef = np.block([[top_left, top_right],
                         [bottom_left, bottom_right]])
    
    # Right-hand side
    Xty = X.T @ y
    Zty = Z.T @ y
    rhs = np.vstack([Xty, Zty])
    
    # Solve MME
    try:
        solution = solve(MME_coef, rhs)
    except np.linalg.LinAlgError:
        # Use pseudo-inverse if singular
        solution = np.linalg.pinv(MME_coef) @ rhs
    
    # Extract EBV (random effects)
    ebv = solution[n_fixed:, 0]
    
    return ebv, all_ids


print("Pedigree BLUP helper functions defined.")

## Create Founders

Generate the initial founder population with haplotypes and set up simulation parameters.

In [ ]:
print("Creating founders...")

# Create founder population
founder_pop = runMacs2(
    nInd=n_parents,
    nChr=n_chr,
    segSites=n_qtl + n_snp,
    genLen=gen_len,
    mutRate=mut_rate
)

print(f"✓ Created founder population: {founder_pop.n_ind} individuals")

# Set simulation parameters
SP = SimParam(founder_pop)

# Restrict segregating sites (separate QTL and SNP)
SP.restrSegSites(minQtlPerChr=n_qtl, minSnpPerChr=n_snp)

# Add SNP chip
if n_snp > 0:
    SP.addSnpChip(n_snp)
    print(f"✓ Added SNP chip: {SP.n_snp_chips} SNP chips")

# Add traits: trait represents yield
# Using addTraitADG for additive, dominance, and GxE effects
SP.addTraitADG(
    nQtlPerChr=n_qtl,
    mean=init_mean_g,
    var=init_var_g,
    varGxE=init_var_ge
)
print(f"✓ Added TraitADG: {SP.n_traits} traits")

# Collect pedigree
SP.setTrackPed(True)
print("✓ Enabled pedigree tracking")

# Create founder parents
Parents = newPop(founder_pop, sim_param=SP)
print(f"✓ Created founder parents: {Parents.n_ind} individuals, {Parents.n_traits} traits")

# Set a phenotype to founder parents
Parents = setPheno(Parents, varE=var_e, reps=rep_ect, simParam=SP)

print(f"\nFounder population summary:")
print(f"  Mean genetic value: {meanG(Parents)[0]:.3f}")
print(f"  Genetic variance: {varG(Parents)[0]:.3f}")

## Fill Breeding Pipeline

Set up the initial breeding pipeline with 16 stages representing different evaluation years.
The pipeline includes:
- Stage 1: Crossing block (F1)
- Stages 2-4: Seedling evaluation (HPT1-3)
- Stages 5-9: Advanced clonal trials (ACT1-5)
- Stages 10-15: Elite clonal trials (ECT1-6)

**Note**: Year effects (p parameter) are not yet supported in AlphaSimPy's `setPheno` function.
The GxE variance is still included in the trait definition, which affects genetic values.

In [ ]:
print("Filling breeding pipeline...")

# Set initial yield trials with unique individuals
# Sample year effects
P = np.random.uniform(size=16)

# Breeding program
for cohort in range(1, 17):
    print(f"  FillPipeline stage: {cohort} of 16")
    
    # Stage 1: Crossing block
    F1 = randCross(Parents, nCrosses=n_crosses, nProgeny=n_progeny, simParam=SP)
    
    if cohort < 16:
        # Stage 2: Germinate the seedlings in the nursery
        Seedlings = setPheno(F1, varE=var_e, reps=rep_hpt, simParam=SP)
    
    if cohort < 15:
        # Stage 3: Plant in the seedlings in the field as HPT and record yields
        HPT1 = Seedlings
    
    if cohort < 14:
        # Stage 4: Record the HPT yields
        HPT2 = HPT1
    
    if cohort < 13:
        # Stage 5: Record the HPT yields
        HPT3 = setPheno(HPT2, varE=var_e, reps=rep_hpt, simParam=SP)
    
    if cohort < 12:
        # Stage 6: Select 500 superior individuals and plant as advanced clonal trials (ACT)
        ACT1 = selectInd(HPT3, nInd=n_clones_act, use="pheno", simParam=SP)
    
    if cohort < 11:
        # Stage 7: Record ACT yields
        ACT2 = ACT1
    
    if cohort < 10:
        # Stage 8: Record ACT yields
        ACT3 = ACT2
    
    if cohort < 9:
        # Stage 9: Record ACT yields
        ACT4 = ACT3
    
    if cohort < 8:
        # Stage 10: Record ACT yields
        ACT5 = setPheno(ACT4, varE=var_e, reps=rep_act, simParam=SP)
    
    if cohort < 7:
        # Stage 11: Select 40 superior individuals and plant as elite clonal trials (ECT)
        ECT1 = selectInd(ACT5, nInd=n_clones_ect, use="pheno", simParam=SP)
    
    if cohort < 6:
        # Stage 12: Record ECT yields
        ECT2 = ECT1
    
    if cohort < 5:
        # Stage 13: Record ECT yields
        ECT3 = ECT2
    
    if cohort < 4:
        # Stage 14: Record ECT yields
        ECT4 = ECT3
    
    if cohort < 3:
        # Stage 15: Record ECT yields
        ECT5 = ECT4
    
    if cohort < 2:
        # Stage 16: Record ECT yields
        ECT6 = setPheno(ECT5, varE=var_e, reps=rep_ect, simParam=SP)

print("\nPipeline filled successfully!")

## Main Simulation Loop

Run the breeding program simulation with burn-in and future phases.
In the future phase, pedigree BLUP is used to predict breeding values and skip HPT stages.

In [ ]:
# Create list to store results from reps
results = []

# Initialize pedigree population dataframe
ped_pop = None

for REP in range(1, n_reps + 1):
    print(f"Working on REP: {REP}")
    
    # Create a data frame to track key parameters
    output = {
        'year': list(range(1, n_cycles + 1)),
        'rep': [REP] * n_cycles,
        'scenario': [scenario_name] * n_cycles,
        'meanG': [0.0] * n_cycles,
        'varG': [0.0] * n_cycles,
        'accSel': [0.0] * n_cycles
    }
    
    # Simulate year effects
    P = np.random.uniform(size=n_cycles)
    
    # Burn-in phase
    for year in range(1, n_burnin + 1):
        print(f"  Working on burnin year: {year}")
        
        # Update parents (pick new parents)
        Parents = selectInd(ECT6, nInd=n_parents, use="pheno", simParam=SP)
        
        # Advance year (advances yield trials by a year and collects records)
        # Stage 16
        ECT6 = setPheno(ECT5, varE=var_e, reps=rep_ect, simParam=SP)
        
        # Stage 15
        ECT5 = ECT4
        
        # Stage 14
        ECT4 = ECT3
        
        # Stage 13
        ECT3 = ECT2
        
        # Stage 12
        ECT2 = ECT1
        
        # Stage 11
        ECT1 = selectInd(ACT5, nInd=n_clones_ect, use="pheno", simParam=SP)
        
        # Stage 10
        ACT5 = setPheno(ACT4, varE=var_e, reps=rep_act, simParam=SP)
        
        # Stage 9
        ACT4 = ACT3
        
        # Stage 8
        ACT3 = ACT2
        
        # Stage 7
        ACT2 = ACT1
        
        # Stage 6
        # Calculate accuracy based on 2000 inds (n_crosses * n_progeny)
        if HPT3.n_ind > 0:
            acc_sel = np.corrcoef(HPT3.gv[:, 0], HPT3.pheno[:, 0])[0, 1]
            output['accSel'][year-1] = acc_sel
        ACT1 = selectInd(HPT3, nInd=n_clones_act, use="pheno", simParam=SP)
        
        # Stage 5
        HPT3 = setPheno(HPT2, varE=var_e, reps=rep_hpt, simParam=SP)
        
        # Stage 4
        HPT2 = HPT1
        
        # Stage 3
        HPT1 = Seedlings
        
        # Stage 2
        Seedlings = setPheno(F1, varE=var_e, reps=rep_hpt, simParam=SP)
        
        # Stage 1: Crossing block
        F1 = randCross(Parents, nCrosses=n_crosses, nProgeny=n_progeny, simParam=SP)
        
        # Store training population (for pedigree records)
        if year >= start_records:
            # Create pedigree dataframe entry for ACT5
            act5_ped = pd.DataFrame({
                'Ind': [int(id_val) for id_val in ACT5.id],
                'Sire': [int(f) if f != '0' else 0 for f in ACT5.father],
                'Dam': [int(m) if m != '0' else 0 for m in ACT5.mother],
                'Year': [year] * ACT5.n_ind,
                'Stage': ['ACT5'] * ACT5.n_ind,
                'Pheno': ACT5.pheno[:, 0].tolist(),
                'GV': ACT5.gv[:, 0].tolist()
            })
            
            if ped_pop is None:
                ped_pop = act5_ped.copy()
            else:
                ped_pop = pd.concat([ped_pop, act5_ped], ignore_index=True)
        
        # Report results
        output['meanG'][year-1] = meanG(Seedlings)[0]
        output['varG'][year-1] = varG(Seedlings)[0]
    
    # Future phase
    # Replace three early stages with pedigree prediction
    # HPT1, HPT2, HPT3 are removed - we use pedigree BLUP instead
    
    for year in range(n_burnin + 1, n_burnin + n_future + 1):
        print(f"  Working on future year: {year}")
        
        # Stage 1: Crossing block
        F1 = randCross(Parents, nCrosses=n_crosses, nProgeny=n_progeny, simParam=SP)
        
        # Stage 2: Seedlings (no phenotype yet)
        Seedlings = F1  # Just assign, no phenotype
        
        # Run pedigree model to predict EBV for Seedlings
        # Prepare prediction dataset for seedlings
        seedlings_ped = pd.DataFrame({
            'Ind': [int(id_val) for id_val in Seedlings.id],
            'Sire': [int(f) if f != '0' else 0 for f in Seedlings.father],
            'Dam': [int(m) if m != '0' else 0 for m in Seedlings.mother],
            'Year': [year] * Seedlings.n_ind,
            'Stage': ['Seedlings'] * Seedlings.n_ind,
            'Pheno': [np.nan] * Seedlings.n_ind,
            'GV': Seedlings.gv[:, 0].tolist()
        })
        
        # Combine with existing pedigree data
        ped_pop_tmp = pd.concat([ped_pop, seedlings_ped], ignore_index=True)
        
        # Prepare pedigree dataframe for BLUP
        ped_df = ped_pop_tmp[['Ind', 'Sire', 'Dam']].drop_duplicates()
        
        # Prepare phenotype data (only individuals with phenotypes)
        pheno_data = ped_pop_tmp[ped_pop_tmp['Pheno'].notna()][['Ind', 'Pheno', 'Year']].copy()
        
        # Run pedigree BLUP
        if len(pheno_data) > 0:
            try:
                ebv, id_order = solve_pedigree_blup(pheno_data, ped_df, var_e=var_e, var_a=init_var_g)
                
                # Map EBV to Seedlings
                id_to_ebv = dict(zip(id_order, ebv))
                seedlings_ebv = np.array([id_to_ebv.get(int(id_val), 0.0) for id_val in Seedlings.id])
                
                # Assign EBV to Seedlings (resize ebv array if needed)
                if Seedlings.ebv.shape[1] == 0:
                    Seedlings.ebv = seedlings_ebv.reshape(-1, 1)
                else:
                    Seedlings.ebv[:, 0] = seedlings_ebv
                
                # Calculate accuracy
                if len(seedlings_ebv) > 1:
                    acc_sel = np.corrcoef(Seedlings.gv[:, 0], seedlings_ebv)[0, 1]
                    output['accSel'][year-1] = acc_sel
                
            except Exception as e:
                print(f"    Warning: Pedigree BLUP failed in year {year}: {e}")
                # Fallback: use phenotypic selection
                Seedlings = setPheno(Seedlings, varE=var_e, reps=rep_hpt, simParam=SP)
                acc_sel = np.corrcoef(Seedlings.gv[:, 0], Seedlings.pheno[:, 0])[0, 1] if Seedlings.n_ind > 1 else 0.0
                output['accSel'][year-1] = acc_sel
        else:
            # No phenotype data yet, use phenotypic selection
            Seedlings = setPheno(Seedlings, varE=var_e, reps=rep_hpt, simParam=SP)
            acc_sel = np.corrcoef(Seedlings.gv[:, 0], Seedlings.pheno[:, 0])[0, 1] if Seedlings.n_ind > 1 else 0.0
            output['accSel'][year-1] = acc_sel
        
        # Stage 6: Select using EBV (instead of HPT3 phenotype)
        ACT1 = selectInd(Seedlings, nInd=n_clones_act, use="ebv", simParam=SP)
        
        # Stage 7
        ACT2 = ACT1
        
        # Stage 8
        ACT3 = ACT2
        
        # Stage 9
        ACT4 = ACT3
        
        # Stage 10
        ACT5 = setPheno(ACT4, varE=var_e, reps=rep_act, simParam=SP)
        
        # Stage 11
        ECT1 = selectInd(ACT5, nInd=n_clones_ect, use="pheno", simParam=SP)
        
        # Stage 12
        ECT2 = ECT1
        
        # Stage 13
        ECT3 = ECT2
        
        # Stage 14
        ECT4 = ECT3
        
        # Stage 15
        ECT5 = ECT4
        
        # Stage 16
        ECT6 = setPheno(ECT5, varE=var_e, reps=rep_ect, simParam=SP)
        
        # Update parents
        Parents = selectInd(ECT6, nInd=n_parents, use="pheno", simParam=SP)
        
        # Store training population (update ped_pop)
        act5_ped = pd.DataFrame({
            'Ind': [int(id_val) for id_val in ACT5.id],
            'Sire': [int(f) if f != '0' else 0 for f in ACT5.father],
            'Dam': [int(m) if m != '0' else 0 for m in ACT5.mother],
            'Year': [year] * ACT5.n_ind,
            'Stage': ['ACT'] * ACT5.n_ind,
            'Pheno': ACT5.pheno[:, 0].tolist(),
            'GV': ACT5.gv[:, 0].tolist()
        })
        
        ped_pop = pd.concat([ped_pop, act5_ped], ignore_index=True)
        
        # Report results
        output['meanG'][year-1] = meanG(Seedlings)[0]
        output['varG'][year-1] = varG(Seedlings)[0]
    
    # Save results from current replicate
    results.append(output)

print("\nSimulation completed!")

## Analyze Results

Visualize the results from the simulation.

In [ ]:
# Combine results from all replicates
df = pd.DataFrame(results[0])  # For single replicate, convert dict to DataFrame

# If multiple replicates, combine them
if len(results) > 1:
    df = pd.concat([pd.DataFrame(r) for r in results], ignore_index=True)

print("Results summary:")
print(df.head(10))
print(f"\nTotal years simulated: {len(df)}")

In [ ]:
# Plotting function
def plot_results(x, y, title, xlabel, ylabel, ylim=None):
    plt.plot(x, y, 'b-', linewidth=2)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    if ylim is not None:
        plt.ylim(ylim)
    plt.grid(True, linestyle='--', alpha=0.7)

# Create plots
fig, axes = plt.subplots(3, 1, figsize=(6, 12))

# Genetic Gain
plt.sca(axes[0])
plot_results(df['year'], df['meanG'], 
              'Genetic gain', 'Year', 'Yield')
plt.axvline(x=n_burnin, color='r', linestyle='--', alpha=0.5, label='Burn-in/Future')
plt.legend()

# Genetic Variance
plt.sca(axes[1])
plot_results(df['year'], df['varG'], 
              'Genetic variance', 'Year', 'Variance')
plt.axvline(x=n_burnin, color='r', linestyle='--', alpha=0.5, label='Burn-in/Future')
plt.legend()

# Selection Accuracy
plt.sca(axes[2])
plot_results(df['year'], df['accSel'], 
              'Selection accuracy', 'Year', 'Correlation')
plt.axvline(x=n_burnin, color='r', linestyle='--', alpha=0.5, label='Burn-in/Future')
plt.legend()

plt.tight_layout()
plt.savefig('PedigreeSelection_Results.png', dpi=150, bbox_inches='tight')
plt.show()

print("Results plot saved as 'PedigreeSelection_Results.png'")

## Summary

This tutorial demonstrated:

1. **Founder Population Creation**: Using `runMacs2` to generate initial haplotypes
2. **Trait Definition**: Adding traits with additive, dominance, and GxE effects using `addTraitADG`
3. **Breeding Pipeline**: Setting up a 16-stage clonal breeding pipeline with:
   - Seedling evaluation (HPT stages)
   - Advanced clonal trials (ACT stages)
   - Elite clonal trials (ECT stages)
4. **Pedigree BLUP**: Using pedigree-based BLUP to predict breeding values and skip early evaluation stages
5. **Burn-in Phase**: Establishing the breeding program with phenotypic selection
6. **Future Phase**: Using pedigree BLUP to accelerate selection by skipping HPT1-3 stages
7. **Genetic Progress**: Tracking genetic gain, variance, and selection accuracy over time

The simulation shows how pedigree-based selection can be used in clonal breeding programs to:
- **Accelerate breeding cycles** by skipping early evaluation stages
- **Improve selection accuracy** by leveraging pedigree relationships
- **Maintain genetic diversity** while achieving genetic gain

Key differences from phenotypic selection:
- **Pedigree BLUP** replaces phenotypic evaluation in early stages (HPT1-3)
- **EBV-based selection** is used instead of phenotype-based selection for Seedlings
- **Training population** accumulates over time to improve BLUP predictions